# LeetCode 1552 — Magnetic Force Between Two Balls bằng Python

Link đề: https://leetcode.com/problems/magnetic-force-between-two-balls/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maxDistance(self, position: List[int], m: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận dạng dạng bài **"tối đa hoá giá trị nhỏ nhất"** (maximize the minimum) và biết vì sao nó gợi ý **tìm kiếm nhị phân trên đáp án**.
2. Viết hàm kiểm tra tham lam `can(d)`: "có đặt được `m` quả bóng sao cho mọi khoảng cách ≥ `d` không?" và chứng minh tham lam "đặt sớm nhất có thể" là tối ưu.
3. Hiểu tính **đơn điệu**: `can(d)` đúng ⇒ `can(d − 1)` đúng — điều kiện để tìm nhị phân được.
4. Viết đúng template tìm nhị phân "tìm giá trị lớn nhất thoả điều kiện" mà không bị lặp vô hạn hay lệch 1.
5. So sánh với baseline: thử mọi tổ hợp, thử mọi khoảng cách tuyến tính.

## 2. Đề bài & ví dụ

Có `n` giỏ, giỏ thứ `i` nằm ở vị trí `position[i]` trên một đường thẳng. Cần bỏ `m` quả bóng vào `m` giỏ khác nhau. Lực từ giữa hai quả bóng ở vị trí `x` và `y` là `|x − y|`. Hãy phân bóng sao cho **lực nhỏ nhất giữa hai quả bóng bất kỳ là lớn nhất**, và trả về giá trị lực đó.

Nói cách khác: chọn `m` vị trí trong `position`, gọi `D` = khoảng cách nhỏ nhất giữa hai vị trí được chọn; tìm cách chọn để `D` lớn nhất.

**Ví dụ 1:** `position = [1,2,3,4,7]`, `m = 3` → `3`. Đặt bóng ở `1, 4, 7`: các lực `3, 3, 6`, nhỏ nhất là `3`. Không thể tốt hơn.

**Ví dụ 2:** `position = [5,4,3,2,1,1000000000]`, `m = 2` → `999999999`. Đặt ở `1` và `1000000000`.

**Ràng buộc:** `2 ≤ n ≤ 10⁵`, `1 ≤ position[i] ≤ 10⁹`, các vị trí đôi một khác nhau, `2 ≤ m ≤ n`.

In [1]:
import bisect
import random
import time
from itertools import combinations
from typing import List

examples = [([1, 2, 3, 4, 7], 3, 3), ([5, 4, 3, 2, 1, 1000000000], 2, 999999999)]
for pos, m, want in examples:
    print(f"position = {pos}, m = {m} -> mong đợi {want}")

position = [1, 2, 3, 4, 7], m = 3 -> mong đợi 3
position = [5, 4, 3, 2, 1, 1000000000], m = 2 -> mong đợi 999999999


## 3. Kiến thức nền

### 3.1 Sắp xếp `sorted` / `list.sort`

Vị trí trong đề **không** được sắp sẵn (ví dụ 2). Khi đã sắp tăng dần, khoảng cách nhỏ nhất giữa các bóng chính là khoảng cách nhỏ nhất giữa **hai bóng liền kề**.

In [2]:
pos = [5, 4, 3, 2, 1, 1000000000]
print(sorted(pos))
chosen = sorted([7, 1, 4])
print("khoảng cách liền kề:", [b - a for a, b in zip(chosen, chosen[1:])], "-> min =", min(b - a for a, b in zip(chosen, chosen[1:])))

[1, 2, 3, 4, 5, 1000000000]
khoảng cách liền kề: [3, 3] -> min = 3


### 3.2 `itertools.combinations` — liệt kê mọi cách chọn

`combinations(arr, m)` sinh mọi bộ `m` phần tử (giữ thứ tự xuất hiện). Dùng cho brute force khi `n` nhỏ.

In [3]:
for comb in combinations([1, 2, 3, 4], 3):
    print(comb)

(1, 2, 3)
(1, 2, 4)
(1, 3, 4)
(2, 3, 4)


### 3.3 `bisect.bisect_left` — tìm vị trí đầu tiên ≥ x

Trên list đã sắp, `bisect_left(a, x)` trả chỉ số nhỏ nhất `i` sao cho `a[i] >= x` (bằng `len(a)` nếu không có). Dùng để "nhảy" tới giỏ kế tiếp đủ xa.

In [4]:
a = [1, 2, 3, 4, 7]
for x in [0, 3, 5, 7, 8]:
    i = bisect.bisect_left(a, x)
    print(f"x={x}: i={i}", "->", a[i] if i < len(a) else "không có")

x=0: i=0 -> 1
x=3: i=2 -> 3
x=5: i=4 -> 7
x=7: i=4 -> 7
x=8: i=5 -> không có


### 3.4 Tìm kiếm nhị phân trên đáp án

Nếu một điều kiện `ok(d)` có dạng `True, True, ..., True, False, ..., False` khi `d` tăng, ta tìm được `d` lớn nhất có `ok(d) = True` trong `O(log(phạm vi))` lần gọi `ok`. Template:

```python
lo, hi = thấp_nhất_chắc_đúng, cao_nhất_có_thể
while lo < hi:
    mid = (lo + hi + 1) // 2   # làm tròn LÊN vì gán lo = mid
    if ok(mid): lo = mid
    else:       hi = mid - 1
return lo
```

In [5]:
def last_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

# ví dụ: số nguyên lớn nhất có bình phương <= 50
print(last_true(0, 50, lambda d: d * d <= 50))

7


## 4. Ý tưởng nền tảng / trực giác

Lấy ví dụ 1 đã sắp: `[1, 2, 3, 4, 7]`, `m = 3`. Liệt kê tay vài cách chọn:

| Chọn | Khoảng cách liền kề | Min |
|------|---------------------|-----|
| 1, 2, 3 | 1, 1 | 1 |
| 1, 3, 7 | 2, 4 | 2 |
| 1, 4, 7 | 3, 3 | **3** |
| 2, 4, 7 | 2, 3 | 2 |
| 1, 2, 7 | 1, 5 | 1 |

Đáp án `3`. Quan sát:

- Bóng đầu tiên nên đặt ở **giỏ trái nhất** (`1`): dời sang phải chỉ làm mất chỗ.
- Bóng cuối thường ở giỏ phải nhất, nhưng điều quan trọng là: **nếu biết trước muốn khoảng cách ≥ d**, ta cứ đặt bóng kế tiếp ở **giỏ sớm nhất cách bóng trước ≥ d**.

Thử hỏi ngược: "có đạt được `D ≥ 4` không?" — đặt `1`, giỏ sớm nhất ≥ `5` là `7`, rồi hết giỏ: chỉ đặt được 2 bóng < 3 ⇒ **không**. "Có đạt `D ≥ 3`?" — `1`, `4`, `7` ⇒ **có**. Vậy đáp án là `3`. Câu hỏi "có/không" dễ hơn nhiều so với câu hỏi "lớn nhất là bao nhiêu".

In [6]:
def greedy_place(sorted_pos, d):
    chosen = [sorted_pos[0]]
    for p in sorted_pos[1:]:
        if p - chosen[-1] >= d:
            chosen.append(p)
    return chosen

sp = sorted(examples[0][0])
for d in range(1, 7):
    ch = greedy_place(sp, d)
    print(f"d={d}: đặt được {len(ch)} bóng tại {ch} -> {'đủ' if len(ch) >= 3 else 'thiếu'} cho m=3")

d=1: đặt được 5 bóng tại [1, 2, 3, 4, 7] -> đủ cho m=3
d=2: đặt được 3 bóng tại [1, 3, 7] -> đủ cho m=3
d=3: đặt được 3 bóng tại [1, 4, 7] -> đủ cho m=3
d=4: đặt được 2 bóng tại [1, 7] -> thiếu cho m=3
d=5: đặt được 2 bóng tại [1, 7] -> thiếu cho m=3
d=6: đặt được 2 bóng tại [1, 7] -> thiếu cho m=3


## 5. Cách 1 — Brute Force / baseline: thử mọi tổ hợp

**Ý tưởng.** Sắp vị trí, duyệt mọi bộ `m` giỏ bằng `combinations`, tính min khoảng cách liền kề, lấy max.

**Độ phức tạp:** `C(n, m) · m` — bùng nổ tổ hợp, chỉ dùng được với `n ≤ ~15`. Dùng làm **chuẩn đối chiếu**.

In [7]:
def brute_combinations(position, m):
    sp = sorted(position)
    best = 0
    for comb in combinations(sp, m):
        d = min(b - a for a, b in zip(comb, comb[1:]))
        best = max(best, d)
    return best

print(brute_combinations([1, 2, 3, 4, 7], 3))
print(brute_combinations([5, 4, 3, 2, 1, 1000000000], 2))

3
999999999


**Baseline thứ hai — thử mọi khoảng cách `d` từ lớn xuống nhỏ.** Đáp án không vượt quá `(max − min) // (m − 1)` (vì `m − 1` khoảng liền kề cộng lại ≤ `max − min`). Thử `d` từ giá trị đó giảm dần, cái đầu tiên tham lam đặt đủ `m` bóng chính là đáp án.

Độ phức tạp `O(n · D)` với `D = (max − min)/(m − 1)` — có thể lên tới `10⁹ · 10⁵`, quá chậm. **Nút thắt:** ta thử `d` **tuyến tính**, trong khi `can(d)` đơn điệu nên có thể **chia đôi** phạm vi.

In [8]:
def can_place(sp, m, d):
    count, last = 1, sp[0]
    for p in sp[1:]:
        if p - last >= d:
            count += 1
            last = p
            if count >= m:
                return True
    return count >= m

def linear_scan(position, m):
    sp = sorted(position)
    d = (sp[-1] - sp[0]) // (m - 1)
    while d > 1 and not can_place(sp, m, d):
        d -= 1
    return d

print(linear_scan([1, 2, 3, 4, 7], 3))
print(linear_scan([1, 3, 6, 10, 15, 21], 4))

3
5


In [9]:
# Minh hoạ nút thắt: số lần gọi can_place khi phạm vi lớn
rng = random.Random(0)
pos_demo = sorted(rng.sample(range(1, 2 * 10**5), 300))
calls = 0
sp = pos_demo
d = (sp[-1] - sp[0]) // (50 - 1)
while d > 1 and not can_place(sp, 50, d):
    d -= 1
    calls += 1
print("đáp án", d, "| số lần thử tuyến tính:", calls)

đáp án 3367 | số lần thử tuyến tính: 704


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — đổi bài tối ưu thành bài quyết định.** Thay vì hỏi "`D` lớn nhất bao nhiêu", hỏi "với `d` cho trước, có đặt được `m` bóng mà mọi khoảng cách ≥ `d` không?" (hàm `can(d)`).

**Insight 2 — `can(d)` giải bằng tham lam `O(n)`.** Sắp vị trí, đặt bóng đầu tại giỏ trái nhất; với mỗi giỏ tiếp theo, *nếu* nó cách bóng vừa đặt ≥ `d` *thì* đặt bóng vào. Đếm số bóng đặt được; `can(d) = (số bóng ≥ m)`.

**Insight 3 — đơn điệu.** *Nếu* đặt được với khoảng cách tối thiểu `d` *thì* cũng đặt được với mọi `d' < d` (cùng cách đặt đó). Dãy `can(1), can(2), ...` có dạng `T, T, ..., T, F, F, ...` ⇒ tìm nhị phân điểm chuyển cuối cùng của `T`.

**Insight 4 — phạm vi.** `can(1)` luôn đúng (vị trí khác nhau, `m ≤ n`). Cận trên `hi = (max − min) // (m − 1)`.

**Invariant khi tìm nhị phân:** luôn có `can(lo) = True` và đáp án ∈ `[lo, hi]`.

In [10]:
sp = sorted(examples[0][0])
print("can(d) với d = 1..7:", [can_place(sp, 3, d) for d in range(1, 8)])
sp2 = sorted([1, 3, 6, 10, 15, 21])
print("can(d) với m=4:", [can_place(sp2, 4, d) for d in range(1, 9)])

can(d) với d = 1..7: [True, True, True, False, False, False, False]
can(d) với m=4: [True, True, True, True, True, False, False, False]


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sort + tìm nhị phân trên đáp án + tham lam

Từng đoạn:
1. `position.sort()` — để duyệt trái → phải.
2. `can(d)` — tham lam, dừng sớm khi đủ `m` bóng.
3. `lo, hi = 1, (sp[-1] − sp[0]) // (m − 1)` rồi template "last true".

In [11]:
def max_distance_bs(position, m):
    sp = sorted(position)

    def can(d):
        count, last = 1, sp[0]
        for p in sp[1:]:
            if p - last >= d:
                count += 1
                last = p
                if count == m:
                    return True
        return False

    lo, hi = 1, (sp[-1] - sp[0]) // (m - 1)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if can(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for pos, m, want in examples:
    print(max_distance_bs(pos, m), "mong đợi", want)

3 mong đợi 3
999999999 mong đợi 999999999


Code nộp LeetCode cho cách 2:

In [12]:
class SolutionBS:
    def maxDistance(self, position: List[int], m: int) -> int:
        position.sort()

        def can(d: int) -> bool:
            count, last = 1, position[0]
            for p in position[1:]:
                if p - last >= d:
                    count += 1
                    last = p
                    if count == m:
                        return True
            return False

        lo, hi = 1, (position[-1] - position[0]) // (m - 1)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if can(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

print([SolutionBS().maxDistance(list(p), m) for p, m, _ in examples])

[3, 999999999]


**Độ phức tạp:** sắp xếp `O(n log n)`; mỗi lần `can` là `O(n)`, có `O(log D)` lần với `D ≤ 10⁹` (~30 lần) ⇒ tổng `O(n log n + n log D)`. Space `O(1)` thêm (ngoài sort).

### 7.2 Cách 3 — Tham lam bằng `bisect` (nhảy giỏ)

Khi `m` nhỏ so với `n`, thay vì duyệt từng giỏ, ta **nhảy** thẳng tới giỏ đầu tiên có vị trí ≥ `last + d` bằng `bisect_left`. Mỗi lần `can` chỉ tốn `O(m log n)`.

Từng đoạn:
1. `i = 0`, `last = sp[0]`, đã đặt 1 bóng.
2. Lặp `m − 1` lần: `i = bisect_left(sp, last + d, i + 1)`; nếu `i == n` ⇒ thất bại.
3. Nếu đủ `m − 1` lần nhảy ⇒ thành công.

In [13]:
def max_distance_bisect(position, m):
    sp = sorted(position)
    n = len(sp)

    def can(d):
        i = 0
        for _ in range(m - 1):
            i = bisect.bisect_left(sp, sp[i] + d, i + 1)
            if i == n:
                return False
        return True

    lo, hi = 1, (sp[-1] - sp[0]) // (m - 1)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if can(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for pos, m, want in examples:
    print(max_distance_bisect(pos, m), "mong đợi", want)

3 mong đợi 3
999999999 mong đợi 999999999


Code nộp LeetCode cho cách 3:

In [14]:
class SolutionBisect:
    def maxDistance(self, position: List[int], m: int) -> int:
        position.sort()
        n = len(position)

        def can(d: int) -> bool:
            i = 0
            for _ in range(m - 1):
                i = bisect.bisect_left(position, position[i] + d, i + 1)
                if i == n:
                    return False
            return True

        lo, hi = 1, (position[-1] - position[0]) // (m - 1)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if can(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

print([SolutionBisect().maxDistance(list(p), m) for p, m, _ in examples])

[3, 999999999]


**Độ phức tạp:** `O(n log n + min(n, m log n) · log D)`. Khi `m` gần `n`, cách 2 nhanh hơn (duyệt tuyến tính rẻ hơn `m` lần bisect); khi `m` nhỏ, cách 3 nhanh hơn.

### 7.3 Cách 4 (gọn) — `bisect` có sẵn với `key` (Python 3.10+)

Python 3.10 cho phép `bisect_left(range(...), True, key=...)` để tìm vị trí đầu tiên mà điều kiện chuyển thành `True`. Ta tìm `d` đầu tiên **không** đặt được, rồi trừ 1. Ngắn nhưng khó đọc hơn; nên hiểu template tay trước.

In [15]:
def max_distance_key(position, m):
    sp = sorted(position)
    hi = (sp[-1] - sp[0]) // (m - 1)
    # tìm d nhỏ nhất trong [1, hi+1] mà can_place(d) == False
    first_false = bisect.bisect_left(range(1, hi + 2), True, key=lambda d: not can_place(sp, m, d)) + 1
    return first_false - 1

print([max_distance_key(p, m) for p, m, _ in examples])

[3, 999999999]


## 8. TRACE từng bước

Trace cách 2 trên ví dụ 1: `sp = [1, 2, 3, 4, 7]`, `m = 3`, `lo = 1`, `hi = (7 − 1) // 2 = 3`.

| Vòng | lo | hi | mid | Tham lam đặt | can(mid) | Cập nhật |
|------|----|----|-----|--------------|----------|----------|
| 1 | 1 | 3 | 2 | 1, 3, (dừng khi đủ 3) → 1,3,7 | True | lo = 2 |
| 2 | 2 | 3 | 3 | 1, 4, 7 | True | lo = 3 |
| — | 3 | 3 | | | | dừng, trả 3 |

In [16]:
def trace_binary_search(position, m):
    sp = sorted(position)
    lo, hi = 1, (sp[-1] - sp[0]) // (m - 1)
    print(f"sp = {sp}, m = {m}, lo = {lo}, hi = {hi}")
    step = 0
    while lo < hi:
        step += 1
        mid = (lo + hi + 1) // 2
        placed = greedy_place(sp, mid)
        ok = len(placed) >= m
        print(f"  vòng {step}: lo={lo}, hi={hi}, mid={mid}, tham lam đặt {placed} -> can={ok}")
        if ok:
            lo = mid
        else:
            hi = mid - 1
    print(f"  dừng: lo = hi = {lo}")
    return lo

trace_binary_search([1, 2, 3, 4, 7], 3)

sp = [1, 2, 3, 4, 7], m = 3, lo = 1, hi = 3
  vòng 1: lo=1, hi=3, mid=2, tham lam đặt [1, 3, 7] -> can=True
  vòng 2: lo=2, hi=3, mid=3, tham lam đặt [1, 4, 7] -> can=True
  dừng: lo = hi = 3


3

In [17]:
trace_binary_search([1, 3, 6, 10, 15, 21, 28], 4)

sp = [1, 3, 6, 10, 15, 21, 28], m = 4, lo = 1, hi = 9
  vòng 1: lo=1, hi=9, mid=5, tham lam đặt [1, 6, 15, 21, 28] -> can=True
  vòng 2: lo=5, hi=9, mid=7, tham lam đặt [1, 10, 21, 28] -> can=True
  vòng 3: lo=7, hi=9, mid=8, tham lam đặt [1, 10, 21] -> can=False
  dừng: lo = hi = 7


7

Trace riêng hàm tham lam `can(d)` để thấy biến `last`, `count` thay đổi qua từng giỏ:

In [18]:
def trace_can(sp, m, d):
    count, last = 1, sp[0]
    print(f"d={d}: đặt bóng 1 tại {last}")
    for p in sp[1:]:
        gap = p - last
        if gap >= d:
            count += 1
            last = p
            print(f"  giỏ {p}: cách {gap} >= {d} -> đặt bóng {count}")
        else:
            print(f"  giỏ {p}: cách {gap} <  {d} -> bỏ qua")
    print(f"  tổng {count} bóng -> can = {count >= m}")
    return count >= m

trace_can([1, 2, 3, 4, 7], 3, 3)
trace_can([1, 2, 3, 4, 7], 3, 4)

d=3: đặt bóng 1 tại 1
  giỏ 2: cách 1 <  3 -> bỏ qua
  giỏ 3: cách 2 <  3 -> bỏ qua
  giỏ 4: cách 3 >= 3 -> đặt bóng 2
  giỏ 7: cách 3 >= 3 -> đặt bóng 3
  tổng 3 bóng -> can = True
d=4: đặt bóng 1 tại 1
  giỏ 2: cách 1 <  4 -> bỏ qua
  giỏ 3: cách 2 <  4 -> bỏ qua
  giỏ 4: cách 3 <  4 -> bỏ qua
  giỏ 7: cách 6 >= 4 -> đặt bóng 2
  tổng 2 bóng -> can = False


False

## 9. Code nộp LeetCode

In [19]:
class Solution:
    def maxDistance(self, position: List[int], m: int) -> int:
        position.sort()

        def can(d: int) -> bool:
            # Tham lam: đặt bóng ở giỏ sớm nhất cách bóng trước >= d
            count, last = 1, position[0]
            for p in position[1:]:
                if p - last >= d:
                    count += 1
                    last = p
                    if count == m:
                        return True
            return False

        lo, hi = 1, (position[-1] - position[0]) // (m - 1)
        while lo < hi:                      # tìm d lớn nhất có can(d) = True
            mid = (lo + hi + 1) // 2
            if can(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

for pos, m, want in examples:
    got = Solution().maxDistance(list(pos), m)
    print(got, want)
    assert got == want

3 3
999999999 999999999


## 10. Vì sao thuật toán đúng

**(a) Tham lam đúng cho `can(d)`.** Giả sử có một cách đặt hợp lệ bất kỳ `q₁ < q₂ < ... < q_m` (mọi khoảng cách ≥ `d`). Gọi `g₁ < g₂ < ...` là các vị trí tham lam chọn. Chứng minh quy nạp `gₖ ≤ qₖ`:
- `g₁ = min(sp) ≤ q₁`.
- Nếu `gₖ ≤ qₖ` thì `q_{k+1} ≥ qₖ + d ≥ gₖ + d`, tức `q_{k+1}` là một ứng viên hợp lệ cho bước `k+1` của tham lam; tham lam chọn giỏ **sớm nhất** thoả ⇒ `g_{k+1} ≤ q_{k+1}` (và chắc chắn tồn tại).

Do đó tham lam đặt được ít nhất `m` bóng mỗi khi có cách đặt hợp lệ ⇒ `can(d)` chính xác.

**(b) Đơn điệu.** Cách đặt hợp lệ cho `d` cũng hợp lệ cho mọi `d' ≤ d`.

**(c) Tìm nhị phân.** Invariant: `can(lo) = True` (ban đầu `lo = 1` luôn đúng) và mọi `d > hi` đều sai (ban đầu `hi` là cận trên lý thuyết). Mỗi vòng thu hẹp `[lo, hi]` mà giữ invariant; `mid = (lo+hi+1)//2 > lo` nên khoảng luôn co lại ⇒ dừng với `lo = hi` = `d` lớn nhất thoả.

In [20]:
# Kiểm tra bằng thực nghiệm: tham lam luôn khớp "có tổ hợp nào hợp lệ không"
rng = random.Random(10)
for _ in range(300):
    sp = sorted(rng.sample(range(1, 30), rng.randint(2, 9)))
    m = rng.randint(2, len(sp))
    d = rng.randint(1, 15)
    exists = any(min(b - a for a, b in zip(c, c[1:])) >= d for c in combinations(sp, m))
    assert can_place(sp, m, d) == exists
print("tham lam == tồn tại cách đặt: OK trên 300 trường hợp")

tham lam == tồn tại cách đặt: OK trên 300 trường hợp


## 11. Phân tích Time / Space

Gọi `n = len(position)`, `D = (max − min) // (m − 1) ≤ 10⁹`.

| Cách | Time | Space |
|------|------|-------|
| Tổ hợp | `O(C(n,m) · m)` | `O(m)` |
| Quét tuyến tính `d` | `O(n log n + n · D)` | `O(n)` |
| Nhị phân + tham lam | `O(n log n + n log D)` | `O(n)` cho bản sort, `O(1)` nếu sort tại chỗ |
| Nhị phân + bisect | `O(n log n + m log n log D)` | như trên |

Với `n = 10⁵`: `log D ≈ 30` ⇒ khoảng `3·10⁶` phép so sánh — nhanh. Không có khác biệt average/worst đáng kể (tham lam có thể dừng sớm khi đủ `m`).

In [21]:
rng = random.Random(1552)
big = rng.sample(range(1, 10**9 + 1), 10**5)
for name, f in [("BS", lambda p, m: SolutionBS().maxDistance(list(p), m)),
                ("Bisect", lambda p, m: SolutionBisect().maxDistance(list(p), m))]:
    for m in [2, 1000, 10**5]:
        t0 = time.perf_counter()
        r = f(big, m)
        print(f"{name:6s} m={m:>6}: {r:>10} ({time.perf_counter() - t0:.3f}s)")

BS     m=     2:  999950880 (0.117s)


BS     m=  1000:     990873 (0.086s)
BS     m=100000:          1 (0.099s)
Bisect m=     2:  999950880 (0.019s)


Bisect m=  1000:     990873 (0.027s)


Bisect m=100000:          1 (0.302s)


## 12. Bộ test

In [22]:
METHODS = {
    "linear_scan": linear_scan,
    "bs": max_distance_bs,
    "bisect": max_distance_bisect,
    "key": max_distance_key,
    "SolutionBS": lambda p, m: SolutionBS().maxDistance(list(p), m),
    "SolutionBisect": lambda p, m: SolutionBisect().maxDistance(list(p), m),
    "Solution": lambda p, m: Solution().maxDistance(list(p), m),
}
for pos, m, want in examples:
    for name, f in METHODS.items():
        assert f(pos, m) == want, (name, pos, m)
print("ví dụ đề: OK")

ví dụ đề: OK


In [23]:
# edge case
edge = [
    ([1, 2], 2, 1),                       # n = m = 2
    ([1, 10**9], 2, 10**9 - 1),           # khoảng cách lớn nhất
    ([1, 2, 3, 4, 5], 5, 1),              # m = n: phải lấy tất cả
    ([10, 1, 5], 3, 4),                   # chưa sắp
    ([1, 2, 3, 100], 2, 99),
    ([1, 3, 6, 10, 15, 21, 28], 4, 7),
]
for pos, m, want in edge:
    for name, f in METHODS.items():
        assert f(pos, m) == want, (name, pos, m, f(pos, m))
    assert brute_combinations(pos, m) == want
print("edge case: OK")

edge case: OK


In [24]:
# ngẫu nhiên có seed: đối chiếu với brute force tổ hợp
rng = random.Random(2024)
for _ in range(1500):
    n = rng.randint(2, 10)
    lim = rng.choice([12, 50, 1000])
    pos = rng.sample(range(1, lim + 1), n)
    m = rng.randint(2, n)
    ref = brute_combinations(pos, m)
    for name, f in METHODS.items():
        assert f(pos, m) == ref, (name, pos, m)
print("1500 test ngẫu nhiên (seed 2024): mọi cách khớp brute force")

1500 test ngẫu nhiên (seed 2024): mọi cách khớp brute force


In [25]:
# hàm Solution không phụ thuộc thứ tự input
rng = random.Random(7)
for _ in range(200):
    pos = rng.sample(range(1, 500), rng.randint(2, 40))
    m = rng.randint(2, len(pos))
    shuffled = pos[:]
    rng.shuffle(shuffled)
    assert Solution().maxDistance(pos[:], m) == Solution().maxDistance(shuffled, m)
print("bất biến theo hoán vị: OK")

bất biến theo hoán vị: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên sắp xếp `position`

Tham lam "duyệt trái → phải" chỉ đúng khi vị trí tăng dần. Ví dụ 2 có input giảm dần.

In [26]:
def wrong_no_sort(position, m):
    sp = position                         # SAI: không sort
    def can(d):
        count, last = 1, sp[0]
        for p in sp[1:]:
            if p - last >= d:
                count, last = count + 1, p
        return count >= m
    lo, hi = 1, max(1, (max(sp) - min(sp)) // (m - 1))
    while lo < hi:
        mid = (lo + hi + 1) // 2
        lo, hi = (mid, hi) if can(mid) else (lo, mid - 1)
    return lo

for pos, m, want in examples + [([10, 1, 5], 3, 4)]:
    print(pos, m, "| sai:", wrong_no_sort(pos, m), "| đúng:", want)

[1, 2, 3, 4, 7] 3 | sai: 3 | đúng: 3
[5, 4, 3, 2, 1, 1000000000] 2 | sai: 999999995 | đúng: 999999999
[10, 1, 5] 3 | sai: 1 | đúng: 4


Sửa: `position.sort()` (hoặc `sp = sorted(position)`) trước khi làm gì khác.

### Lỗi 2 — `mid = (lo + hi) // 2` trong khi cập nhật `lo = mid`

Khi `hi = lo + 1` và `can(mid)` đúng, `mid = lo` ⇒ `lo` không đổi ⇒ **lặp vô hạn**. Ta chạy với bộ đếm giới hạn để thấy hiện tượng mà không treo notebook.

In [27]:
def wrong_mid(position, m, max_iter=100):
    sp = sorted(position)
    lo, hi = 1, (sp[-1] - sp[0]) // (m - 1)
    it = 0
    while lo < hi:
        it += 1
        if it > max_iter:
            return f"KẸT ở lo={lo}, hi={hi} sau {max_iter} vòng"
        mid = (lo + hi) // 2              # SAI: làm tròn xuống
        if can_place(sp, m, mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print("sai:", wrong_mid([1, 2, 3, 4, 7], 3), "| đúng:", 3)

sai: KẸT ở lo=2, hi=3 sau 100 vòng | đúng: 3


Sửa: `mid = (lo + hi + 1) // 2` khi nhánh đúng là `lo = mid`. (Quy tắc: gán `lo = mid` ⇒ làm tròn lên; gán `hi = mid` ⇒ làm tròn xuống.)

### Lỗi 3 — So sánh `>` thay vì `>=` trong tham lam

Khoảng cách **bằng** `d` vẫn hợp lệ. Dùng `>` làm `can(d)` thực chất kiểm tra `d + 1` ⇒ đáp án nhỏ hơn 1.

In [28]:
def wrong_strict(position, m):
    sp = sorted(position)
    def can(d):
        count, last = 1, sp[0]
        for p in sp[1:]:
            if p - last > d:              # SAI: phải là >=
                count, last = count + 1, p
        return count >= m
    lo, hi = 1, (sp[-1] - sp[0]) // (m - 1)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        lo, hi = (mid, hi) if can(mid) else (lo, mid - 1)
    return lo

for pos, m, want in examples + [([1, 2, 3, 4, 5], 5, 1)]:
    print(pos, m, "| sai:", wrong_strict(pos, m), "| đúng:", want)

[1, 2, 3, 4, 7] 3 | sai: 2 | đúng: 3
[5, 4, 3, 2, 1, 1000000000] 2 | sai: 999999998 | đúng: 999999999
[1, 2, 3, 4, 5] 5 | sai: 1 | đúng: 1


Sửa: `if p - last >= d`. (Với `[1..5], m = 5` lỗi này còn trả `1` "ăn may" vì `lo` khởi tạo là 1 — test kỹ các ví dụ đề mới thấy.)

### Lỗi 4 — Dùng template `while lo <= hi` nhưng trả về `lo`

Với template `while lo <= hi: if ok(mid): lo = mid + 1 else: hi = mid − 1`, khi dừng `hi` là giá trị đúng cuối cùng, còn `lo = hi + 1` là giá trị sai đầu tiên.

In [29]:
def wrong_return(position, m):
    sp = sorted(position)
    lo, hi = 1, (sp[-1] - sp[0]) // (m - 1)
    while lo <= hi:
        mid = (lo + hi) // 2
        if can_place(sp, m, mid):
            lo = mid + 1
        else:
            hi = mid - 1
    return lo                              # SAI: phải trả hi

for pos, m, want in examples:
    print(pos, m, "| sai:", wrong_return(pos, m), "| đúng:", want)

[1, 2, 3, 4, 7] 3 | sai: 4 | đúng: 3
[5, 4, 3, 2, 1, 1000000000] 2 | sai: 1000000000 | đúng: 999999999


Sửa: `return hi` (hoặc lưu `ans = mid` mỗi khi `ok(mid)`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Từ khoá **"maximize the minimum"** hoặc **"minimize the maximum"** ⇒ gần như chắc chắn là *binary search on answer* + hàm kiểm tra tham lam.
- Hỏi ngay: "nếu đáp án là `x` thì kiểm tra khả thi thế nào?" Nếu kiểm tra được `O(n)` và đơn điệu ⇒ xong.
- Cận trên chặt: `(max − min) // (m − 1)`; cận dưới `1`. Phạm vi `10⁹` chỉ tốn ~30 vòng.
- Nhớ cặp quy tắc làm tròn `mid` để khỏi lặp vô hạn.
- Bài "đặt điểm cách nhau ≥ d" ⇒ tham lam đặt sớm nhất; bài "chia mảng thành ≤ k đoạn tổng ≤ S" ⇒ tham lam kéo dài đoạn tối đa.

## 15. Biến thể của bài

1. **Aggressive Cows (SPOJ AGGRCOW)** — y hệt bài này.
2. **Split Array Largest Sum (LC 410)** — "minimize the maximum" tổng đoạn; tìm nhị phân `S`, tham lam đếm số đoạn.
3. **Capacity To Ship Packages Within D Days (LC 1011)** — cùng khuôn với LC 410.
4. **Koko Eating Bananas (LC 875)** — tìm nhị phân tốc độ.
5. **Minimized Maximum of Products Distributed (LC 2064)**, **Maximum Tastiness of Candy Basket (LC 2517)** — LC 2517 gần như trùng bài này (giá kẹo thay cho vị trí).
6. Biến thể: trả về **chính các vị trí** đặt bóng (bài tập 4).

In [30]:
# LC 2517 Maximum Tastiness of Candy Basket chính là bài này với price thay cho position, k thay cho m
def maximum_tastiness(price, k):
    return Solution().maxDistance(list(price), k)

print(maximum_tastiness([13, 5, 1, 8, 21, 2], 3), "(đề LC 2517 ví dụ 1 cho 8)")
print(maximum_tastiness([1, 3, 1], 2), "(ví dụ 2 cho 2; lưu ý LC 2517 cho phép giá trùng)")

8 (đề LC 2517 ví dụ 1 cho 8)
2 (ví dụ 2 cho 2; lưu ý LC 2517 cho phép giá trùng)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1a. Tổ hợp | thử mọi bộ `m` giỏ | `O(C(n,m)·m)` | `O(m)` | kiểm chứng, `n ≤ 15` |
| 1b. Quét `d` tuyến tính | thử `d` giảm dần + tham lam | `O(n·D)` | `O(n)` | phạm vi nhỏ |
| 2. Nhị phân + tham lam tuyến tính | `can(d)` đơn điệu | `O(n log n + n log D)` | `O(1)`–`O(n)` | **bản nộp** |
| 3. Nhị phân + bisect | nhảy giỏ bằng `bisect_left` | `O(n log n + m log n log D)` | như trên | `m` nhỏ, `n` lớn |
| 4. `bisect` với `key` | thư viện tìm điểm chuyển | như cách 2 | như cách 2 | viết nhanh (Python ≥ 3.10) |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxDistance([1,2,3,4,5,6], 3)` và `maxDistance([10,1,100], 2)` trả về gì?

**Bài 2 (tính tay):** Ghi lại dãy `(lo, hi, mid, can)` khi chạy cách 2 trên `position = [1, 4, 9, 16, 25]`, `m = 3`.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_can(sp, m, d)` — hàm kiểm tra tham lam trên mảng đã sắp.

**Bài 4 (tự viết lại):** Viết `ex4_positions(position, m)` trả về **danh sách `m` vị trí** đặt bóng đạt lực tối thiểu lớn nhất.

**Bài 5 (mini challenge):** Viết `ex5_split(nums, k)` (LC 410): chia `nums` (số không âm) thành đúng `k` đoạn liên tiếp khác rỗng sao cho **tổng lớn nhất của các đoạn là nhỏ nhất**; trả về giá trị đó. Dùng tìm nhị phân trên đáp án.

In [31]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_can(sp, m, d):
    count, last = 1, sp[0]
    for p in sp[1:]:
        # TODO: nếu p cách last ít nhất d thì đặt bóng (tăng count, cập nhật last)
        pass
    # TODO: trả True nếu count >= m
    return None

In [32]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_positions(position, m):
    # TODO
    raise NotImplementedError

def ex5_split(nums, k):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Cận trên `(6−1)//2 = 2` cho ví dụ đầu. (2) `d = 2`: `1, 3, 5` ⇒ đủ. (3) Đáp án `2` và `99`.

**Bài 2.** (1) `lo = 1`, `hi = (25 − 1)//2 = 12`. (2) `mid = 7`: tham lam `1, 9, 16` ⇒ True ⇒ `lo = 7`; `mid = 10`: `1, 16` rồi `25 − 16 = 9 < 10` ⇒ False ⇒ `hi = 9`. (3) Tiếp `mid = 8`: `1, 9`, bỏ `16` (vì `16 − 9 = 7 < 8`), lấy `25` ⇒ True ⇒ `lo = 8`; `mid = 9`: `1, 16, 25` ⇒ True ⇒ `lo = 9 = hi`. Đáp án `9`.

**Bài 3.** (1) `if p - last >= d:`. (2) `count += 1; last = p`. (3) `return count >= m`.

**Bài 4.** (1) Tìm đáp án `D` bằng `Solution`. (2) Chạy lại tham lam với `D`, ghi lại vị trí. (3) Cắt lấy `m` vị trí đầu (tham lam có thể đặt được nhiều hơn `m`).

**Bài 5.** (1) Đáp án nằm trong `[max(nums), sum(nums)]`. (2) `need(S)` = số đoạn tối thiểu khi mỗi đoạn tổng ≤ `S` (tham lam kéo dài đoạn). (3) Tìm `S` **nhỏ nhất** có `need(S) ≤ k` (template "first true": `mid = (lo+hi)//2`, `hi = mid`).

## 19. Đáp án tham khảo cho bài tập

In [33]:
# Bài 1
a1 = (Solution().maxDistance([1, 2, 3, 4, 5, 6], 3), Solution().maxDistance([10, 1, 100], 2))
print("Bài 1:", a1)
assert a1 == (2, 99)

# Bài 2
a2 = trace_binary_search([1, 4, 9, 16, 25], 3)
assert a2 == 9 == brute_combinations([1, 4, 9, 16, 25], 3)

Bài 1: (2, 99)
sp = [1, 4, 9, 16, 25], m = 3, lo = 1, hi = 12
  vòng 1: lo=1, hi=12, mid=7, tham lam đặt [1, 9, 16, 25] -> can=True
  vòng 2: lo=7, hi=12, mid=10, tham lam đặt [1, 16] -> can=False
  vòng 3: lo=7, hi=9, mid=8, tham lam đặt [1, 9, 25] -> can=True
  vòng 4: lo=8, hi=9, mid=9, tham lam đặt [1, 16, 25] -> can=True
  dừng: lo = hi = 9


In [34]:
# Bài 3
def ex3_can_sol(sp, m, d):
    count, last = 1, sp[0]
    for p in sp[1:]:
        if p - last >= d:
            count += 1
            last = p
    return count >= m

rng3 = random.Random(3)
for _ in range(500):
    sp = sorted(rng3.sample(range(1, 40), rng3.randint(2, 9)))
    m = rng3.randint(2, len(sp))
    d = rng3.randint(1, 20)
    exists = any(min(b - a for a, b in zip(c, c[1:])) >= d for c in combinations(sp, m))
    assert ex3_can_sol(sp, m, d) == exists
print("Bài 3: OK")

Bài 3: OK


In [35]:
# Bài 4
def ex4_positions_sol(position, m):
    sp = sorted(position)
    D = Solution().maxDistance(list(position), m)
    chosen = [sp[0]]
    for p in sp[1:]:
        if len(chosen) == m:
            break
        if p - chosen[-1] >= D:
            chosen.append(p)
    return chosen

rng4 = random.Random(4)
for _ in range(500):
    pos = rng4.sample(range(1, 60), rng4.randint(2, 10))
    m = rng4.randint(2, len(pos))
    ch = ex4_positions_sol(pos, m)
    assert len(ch) == m and set(ch) <= set(pos)
    assert min(b - a for a, b in zip(ch, ch[1:])) == brute_combinations(pos, m)
print("Bài 4: OK, ví dụ:", ex4_positions_sol([1, 2, 3, 4, 7], 3))

Bài 4: OK, ví dụ: [1, 4, 7]


In [36]:
# Bài 5 (LC 410)
def ex5_split_sol(nums, k):
    def need(S):
        parts, cur = 1, 0
        for x in nums:
            if cur + x > S:
                parts += 1
                cur = x
            else:
                cur += x
        return parts

    lo, hi = max(nums), sum(nums)
    while lo < hi:                         # first true: need(S) <= k
        mid = (lo + hi) // 2
        if need(mid) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

def ex5_brute(nums, k):
    n = len(nums)
    best = None
    for cuts in combinations(range(1, n), k - 1):
        bounds = (0,) + cuts + (n,)
        worst = max(sum(nums[bounds[i]:bounds[i + 1]]) for i in range(k))
        best = worst if best is None else min(best, worst)
    return best

assert ex5_split_sol([7, 2, 5, 10, 8], 2) == 18
assert ex5_split_sol([1, 2, 3, 4, 5], 2) == 9
rng5 = random.Random(5)
for _ in range(600):
    nums = [rng5.randint(0, 20) for _ in range(rng5.randint(1, 8))]
    k = rng5.randint(1, len(nums))
    assert ex5_split_sol(nums, k) == ex5_brute(nums, k), (nums, k)
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def maxDistance(self, position, m):
        position.sort()
        def can(d):
            count, last = 1, position[0]
            for p in position[1:]:
                if p - last >= d:
                    count, last = count + 1, p
                    if count == m:
                        return True
            return False
        lo, hi = 1, (position[-1] - position[0]) // (m - 1)
        while lo < hi:
            mid = (lo + hi + 1) // 2      # last true -> làm tròn lên
            if can(mid): lo = mid
            else:        hi = mid - 1
        return lo
```

**Template "maximize the minimum":** đáp án `x` → hỏi `can(x)` (tham lam), chứng minh đơn điệu, tìm nhị phân "last true". **"Minimize the maximum":** tìm "first true" (`mid = (lo+hi)//2`, `hi = mid`).

**Checklist trước khi submit:**
- [ ] Đã `sort` vị trí.
- [ ] Tham lam dùng `>=` (khoảng cách bằng `d` vẫn hợp lệ).
- [ ] Cặp `mid` / cập nhật khớp nhau (lo = mid ⇒ `+1` khi tính mid).
- [ ] `hi = (max − min) // (m − 1)` (không chia cho 0 vì `m ≥ 2`).
- [ ] Thử nhanh: `([1,2,3,4,7],3) → 3`, `([5,4,3,2,1,10⁹],2) → 999999999`, `([1,2],2) → 1`.